In [ ]:
import pandas as pd
import json
import os
import numpy as np
import re

In [ ]:
df = pd.read_csv("../1_Raspagem_de_Dados/3_metadados_completos_sol_sbc.csv")

In [ ]:
df.info()

### Obtenção do Idioma Geral do Artigo, Introdução e Conclusão

In [ ]:
json_folder = "json"

# Garantir que Date seja datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extrair Ano
df['Year'] = df['Date'].dt.year

# Colunas Auxiliares
df["Paper_Language"] = None

df["has_intro"] = 0
df["has_conclusion"] = 0
df["flag_intro_conc"] = 0

df["Introduction"] = None
df["Conclusion"] = None

df["Area"] = None

INTRO_PATTERNS = [
    "introd"
]

CONCLUSION_PATTERNS = [
    "concl",
    "considerações finais",
    "consideracoes finais",
    "final remarks",
    "concluding"
]

In [ ]:
AREA_MAP = {
    # IA
    r'ENIAC|BRACIS|KDMILE|STIL|BWAIF|ARTIFICIAL INTELLIGENCE|MACHINE LEARNING|ERAMIA|WESAAC':
        'IA',

    # IHC
    r'\bIHC\b|HUMAN-COMPUTER INTERACTION|INTERACTIVE SYSTEMS|CAPAIHC|WEIHC|WAIHCWS|WIDE':
        'IHC',

    # Engenharia de Software
    r'SBES|SBCARS|SAST|VEM|MSSIS|SE4FP|SE4GAMES|SEDT|ISE|WBOTS|SOFTWARE ENGINEERING|OPENSCIENSE|CIBSE|WASHES|BWARE|CBSOFT|SBQS':
        'ES',

    # Redes
    r'SBRC|ERRC|WGRS|WPERFORMANCE|WPEIF|W6G|WQUNETS|WSLICE|WFIBRE|WPIETF|NETWORK|NETWORKS|INTERNET SERVICES':
        'REDES',

    # Segurança
    r'SBSEG|LADC|CYBERSECURITY|FORENSICS|DEPENDABILITY|SAFETY|SECURITY|WTF|RECS|WSENSING|SAFELIFE|WAFERS|STAMP|DIGITAL IDENTITY':
        'SEG',

    # Banco de Dados
    r'SBBD|ERBD|DATA MANAGEMENT|DATABASE':
        'BD',

    # Alto Desempenho
    r'SSCAD|SBAC-PAD|HIGH PERFORMANCE|ERAD':
        'HPC',

    # Educação
    r'CBIE|SBIE|WIE|WEI|EDUCOMP|CTRL\+E|SBC-EB|WAPLA|WAVE|WPCI|WEADEH|WETIE|DESAFIE|WIEI|EDUCATION|EDUCA':
        'EDU',

    # Sistemas de Informação
    r'SBSI|WSIS|INFORMATION SYSTEMS|ISYS':
        'SI',

    # Computação Gráfica / Multimídia
    r'SIBGRAPI|SVR|WVC|WEBMEDIA|IMX|SENSORYX|XR IN GAMES|COMPUTER GRAPHICS|MULTIMEDIA|VIRTUAL|AUGMENTED REALITY|SBCM':
        'CG',

    # Arquitetura / Hardware
    r'SBCCI|SBESC|COMPUTER ARCHITECTURE|INTEGRATED CIRCUITS':
        'ARQ',

    # Teoria da Computação
    r'\bETC\b|WEIT|WBL|SBLP|SBMF|THEORY OF COMPUTATION|FORMAL METHODS|LOGIC':
        'TC',

    # Robótica
    r'SBR/LARS|ROBOTICS':
        'ROB',

    # Saúde
    r'SBCAS|BSB|HEALTH|BIOINFORMATICS|ERCAS':
        'SAUDE',

    # Ubicomp / IoT
    r'SBCUP|UBIQUITOUS|PERVASIVE|WBCI|URBAN COMPUTING|WCGA':
        'UBICOMP',

    # Social / Ética
    r'WIT|WICS|MOSAICO|ETHICS|INCLUSION|DIVERSITY|SOCIAL':
        'SOC',

    # Quântica
    r'QUANTUM|QUNETS':
        'QUANT',

    # Jogos
    r'SBGAMES|WIPLAY|XR IN GAMES|GAME':
        'GAMES',
}

In [ ]:
# FUNÇÃO DE CATEGORIZAÇÃO
def categorize_event(event_name: str) -> str:

    if pd.isna(event_name):
        return 'GERAL'

    event_upper = str(event_name).upper()

    for pattern, area in AREA_MAP.items():

        if re.search(pattern, event_upper):

            return area

    return 'GERAL'

# FUNÇÃO PARA EXTRAÇÃO DE SEÇÕES
def extract_sections(obj, sections=None):

    if sections is None:
        sections = []

    if isinstance(obj, dict):

        # Apenas se o próprio nó possui título
        if "title" in obj:

            title = str(obj["title"]).lower()

            text_parts = []

            # text
            if "text" in obj:

                if isinstance(obj["text"], str):

                    text_parts.append(obj["text"])

            # paragraphs
            if "paragraphs" in obj:

                if isinstance(obj["paragraphs"], list):

                    for p in obj["paragraphs"]:

                        if isinstance(p, str):

                            text_parts.append(p)

                        elif isinstance(p, dict):

                            if "text" in p:

                                text_parts.append(
                                    str(p["text"])
                                )

            # content
            if "content" in obj:

                if isinstance(obj["content"], str):

                    text_parts.append(obj["content"])

            # salvar apenas se houver texto
            if text_parts:

                sections.append({
                    "title": title,
                    "text": "\n".join(text_parts)
                })

        # continuar recursão
        for value in obj.values():

            extract_sections(value, sections)

    elif isinstance(obj, list):

        for item in obj:

            extract_sections(item, sections)

    return sections

In [ ]:
print("=" * 60)
print("PROCESSANDO JSONS")
print("=" * 60)

for file_name in os.listdir(json_folder):

    if file_name.endswith(".json"):

        try:
            # EXTRAIR ID DO JSON
            json_id = int(
                file_name.replace("sbc_", "").replace(".json", "")
            )

            # MAPEAR PARA O DATAFRAME
            # JSON começa em 0
            # DataFrame começa em 1

            df_index = json_id + 1

            print(f"\nProcessando: {file_name}")
            print(f"JSON ID: {json_id}")
            print(f"DF INDEX: {df_index}")

            # ABRIR JSON
            file_path = os.path.join(json_folder,file_name)

            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)


            # IDIOMA
            language = data.get("language", None)

            df.loc[df["index"] == df_index,"Paper_Language"] = language

            # EXTRAIR SEÇÕES
            sections = extract_sections(data)

            introduction_text = None

            conclusion_text = None

            # BUSCAR INTRODUÇÃO E CONCLUSÃO
            for section in sections:

                title = section['title']

                text = section['text']

                # INTRODUÇÃO
                if introduction_text is None:

                    if any(pattern in title for pattern in INTRO_PATTERNS):
                        introduction_text = text

                # CONCLUSÃO
                if conclusion_text is None:

                    if any(
                        pattern in title
                        for pattern in CONCLUSION_PATTERNS
                    ):

                        conclusion_text = text

            # FLAGS
            has_intro = introduction_text is not None

            has_conclusion = conclusion_text is not None

            flag_intro_conc = (has_intro and has_conclusion)

            # SALVAR FLAGS
            df.loc[df["index"] == df_index,"has_intro"] = int(has_intro)

            df.loc[df["index"] == df_index,"has_conclusion"] = int(has_conclusion)

            df.loc[df["index"] == df_index, "flag_intro_conc"] = int(flag_intro_conc)

            # SALVAR TEXTOS
            df.loc[df["index"] == df_index, "Introduction"] = introduction_text

            df.loc[df["index"] == df_index, "Conclusion"] = conclusion_text

            # DEBUG
            print(f"Idioma: {language}")
            print(f"Has Intro: {has_intro}")
            print(f"Has Conclusion: {has_conclusion}")

        except Exception as e:

            print(f"Erro ao processar {file_name}: {e}")



# MAPEAR ÁREAS
df["Area"] = df["Event"].apply(categorize_event)

In [ ]:
df.to_csv('metadados_completos_sol_sbc_com_idioma_area_introducao_conclusao.csv', index=False)

In [ ]:
df.info()

### Divisão das Amostras Válidas por Classe

1. Filtrar papers válidos
2. Calcular distribuição temporal
3. Calcular distribuição por área
4. Realizar amostragem estratificada
5. Dividir entre classes sem reutilização
6. Gerar dataset_control.csv



In [ ]:
# CONFIGURAÇÕES
TARGET_PER_CLASS = 2257

RANDOM_STATE = 42

CLASSES = [
    "Humana",
    "Polida_IA",
    "Gerada"
]

In [ ]:
def texto_valido(x):

    if pd.isna(x):
        return False

    if not isinstance(x, str):
        return False

    if x.strip() == "":
        return False

    return True

In [ ]:
# ETAPA 1 — FILTRAR PAPERS VÁLIDOS
df = pd.read_csv('metadados_completos_sol_sbc_com_idioma_area_introducao_conclusao.csv')

df_filtered = df[
    (df["Year"] >= 2010) &
    (df["Year"] <= 2022) &
    (df["Abstract_Language"] == "pt") &
    (df["Paper_Language"] == "pt") &
    (df["flag_intro_conc"] == 1) &
    (df["Introduction"].apply(texto_valido)) &
    (df["Conclusion"].apply(texto_valido))
].copy()

print(f"Total de papers válidos: {df_filtered.shape[0]}")

In [ ]:
# ETAPA 2 — DISTRIBUIÇÃO TEMPORAL

year_distribution = (
    df_filtered
    .groupby("Year")
    .size()
    .reset_index(name="Count")
)

year_distribution["Year_Percentage"] = (
    year_distribution["Count"] /
    year_distribution["Count"].sum()
)

# Total necessário considerando 3 classes
TOTAL_TARGET = TARGET_PER_CLASS * 3

year_distribution["Target_Total"] = (
    year_distribution["Year_Percentage"] *
    TOTAL_TARGET
).round().astype(int)

print(year_distribution)

In [ ]:
# ETAPA 3 — DISTRIBUIÇÃO POR ÁREA

area_distribution = (
    df_filtered
    .groupby(["Year", "Area"])
    .size()
    .reset_index(name="Count")
)

# Total por ano
year_totals = (
    area_distribution
    .groupby("Year")["Count"]
    .sum()
    .reset_index(name="Year_Total")
)

# Merge
area_distribution = area_distribution.merge(
    year_totals,
    on="Year"
)

# Percentual da área dentro do ano
area_distribution["Area_Percentage"] = (
    area_distribution["Count"] /
    area_distribution["Year_Total"]
)

# Adicionar meta anual
area_distribution = area_distribution.merge(
    year_distribution[
        ["Year", "Target_Total"]
    ],
    on="Year"
)

# Quantidade final do estrato
area_distribution["Target_Stratum"] = (
    area_distribution["Area_Percentage"] *
    area_distribution["Target_Total"]
).round().astype(int)


area_distribution.head()

In [ ]:
# ETAPA 4 — DIVISÃO ENTRE CLASSES

# Cada estrato será dividido igualmente
# entre:
# - Humana
# - Polida_IA
# - Gerada

area_distribution["Per_Class"] = (
    area_distribution["Target_Stratum"] // 3
)

print(
    area_distribution[
        [
            "Year",
            "Area",
            "Target_Stratum",
            "Per_Class"
        ]
    ]
)

In [ ]:
# ETAPA 5 — AMOSTRAGEM ESTRATIFICADA

selected_rows = []

for _, row in area_distribution.iterrows():

    year = row["Year"]

    area = row["Area"]

    per_class = row["Per_Class"]

    candidates = df_filtered[
        (df_filtered["Year"] == year) &
        (df_filtered["Area"] == area)
    ].copy()

    candidates = candidates.sample(
        frac=1,
        random_state=RANDOM_STATE
    )

    total_needed = per_class * 3

    # Não exceder quantidade disponível
    total_needed = min(
        total_needed,
        len(candidates)
    )

    candidates = candidates.iloc[:total_needed]

    split_size = len(candidates) // 3

    splits = [
        candidates.iloc[:split_size],
        candidates.iloc[split_size:split_size * 2],
        candidates.iloc[split_size * 2:]
    ]

    for class_name, split_df in zip(CLASSES, splits):

        split_df = split_df.copy()

        split_df["Class"] = class_name

        selected_rows.append(split_df)

    print(
        f"Ano: {year} | "
        f"Área: {area} | "
        f"Selecionados: {len(candidates)}"
    )

In [ ]:
# ETAPA 6 — DATAFRAME FINAL

df_final = pd.concat(selected_rows)

df_final = df_final.reset_index(drop=True)

# Selecionar apenas colunas relevantes
df_final = df_final[
    [
        "index",
        "Title",
        "Year",
        "Event",
        "Area",
        "Abstract",
        "Introduction",
        "Conclusion",
        "Class"
    ]
]

In [ ]:
# ETAPA 7 — VERIFICAÇÕES FINAIS

print("\nQuantidade por classe:")
print(
    df_final["Class"]
    .value_counts()
)

print("\nQuantidade total:")
print(df_final.shape[0])

print("\nPapers duplicados:")
print(
    df_final["index"]
    .duplicated()
    .sum()
)

print("\nDistribuição temporal:")
print(
    df_final["Year"]
    .value_counts()
    .sort_index()
)

print("\nDistribuição por área:")
print(
    df_final["Area"]
    .value_counts()
)

In [ ]:
df_final.to_csv("dataset_de_amostras.csv",index=False)

### Pré-Processamento Textual da Introdução e Conclusão

In [ ]:
import pandas as pd
df = pd.read_csv('dataset_de_amostras.csv')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6618 entries, 0 to 6617
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   index         6618 non-null   int64 
 1   Title         6618 non-null   object
 2   Year          6618 non-null   int64 
 3   Event         6618 non-null   object
 4   Area          6618 non-null   object
 5   Abstract      6618 non-null   object
 6   Introduction  6618 non-null   object
 7   Conclusion    6618 non-null   object
 8   Class         6618 non-null   object
dtypes: int64(2), object(7)
memory usage: 465.5+ KB


In [ ]:
import unicodedata
import re
import pandas as pd

# =====================================================
# 1. RECONSTRUÇÃO (acentos, cedilha, etc.)
# =====================================================

AGUDO = {
    "a": "á", "e": "é", "i": "í", "o": "ó", "u": "ú",
    "A": "Á", "E": "É", "I": "Í", "O": "Ó", "U": "Ú",
    "ı": "í"
}

TIL = {
    "a": "ã", "o": "õ",
    "A": "Ã", "O": "Õ"
}

CIRCUNFLEXO = {
    "a": "â", "e": "ê", "o": "ô",
    "A": "Â", "E": "Ê", "O": "Ô"
}

CRASE = {
    "a": "à",
    "A": "À"
}

CEDILHA = {
    "c": "ç",
    "C": "Ç"
}

DIACRITICOS = {"́", "̃", "̧"}
PSEUDO = {"ˆ", "`"}


def proxima_letra(texto, inicio):
    j = inicio
    while j < len(texto):
        c = texto[j]
        if c == " ":
            j += 1
            continue
        return j, c
    return None, None


def reconstruir(texto):

    resultado = []
    i = 0

    while i < len(texto):

        char = texto[i]

        # -------------------------
        # cedilha retroativa
        # -------------------------
        if char == "̧":
            if resultado:
                anterior = resultado[-1]
                if anterior in CEDILHA:
                    resultado[-1] = CEDILHA[anterior]
            i += 1
            continue

        # -------------------------
        # agudo
        # -------------------------
        elif char == "́":
            pos, prox = proxima_letra(texto, i + 1)
            if prox in AGUDO:
                resultado.append(AGUDO[prox])
                i = pos + 1
                continue

        # -------------------------
        # til
        # -------------------------
        elif char == "̃":
            pos, prox = proxima_letra(texto, i + 1)
            if prox in TIL:
                resultado.append(TIL[prox])
                i = pos + 1
                continue

        # -------------------------
        # circunflexo
        # -------------------------
        elif char == "ˆ":
            pos, prox = proxima_letra(texto, i + 1)
            if prox in CIRCUNFLEXO:
                resultado.append(CIRCUNFLEXO[prox])
                i = pos + 1
                continue

        # -------------------------
        # crase
        # -------------------------
        elif char == "`":
            pos, prox = proxima_letra(texto, i + 1)
            if prox in CRASE:
                if resultado and resultado[-1] != " ":
                    resultado.append(" ")
                resultado.append(CRASE[prox])
                i = pos + 1
                continue

        # -------------------------
        # espaço artificial
        # -------------------------
        if char == " ":
            if i + 1 < len(texto):
                prox = texto[i + 1]
                if prox in DIACRITICOS or prox in PSEUDO:
                    i += 1
                    continue

        # -------------------------
        # normalização ı
        # -------------------------
        if char == "ı":
            resultado.append("i")
        else:
            resultado.append(char)

        i += 1

    return unicodedata.normalize("NFC", "".join(resultado))


# =====================================================
# 2. LIMPEZA FINAL (Markdown + ruído)
# =====================================================

def limpar_formatacao(texto):

    # remove espaços quebrados de markdown
    texto = re.sub(r"\*\*\s*\*\*", "", texto)
    texto = re.sub(r"\*\s*\*", "", texto)
    texto = re.sub(r"_\s*_", "", texto)

    # markdown normal
    texto = re.sub(r"\*\*(.*?)\*\*", r"\1", texto)
    texto = re.sub(r"\*(.*?)\*", r"\1", texto)
    texto = re.sub(r"_(.*?)_", r"\1", texto)

    # espaços e pontuação
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"\s+([,.;:])", r"\1", texto)

    return texto.strip()


# =====================================================
# 3. PIPELINE FINAL
# =====================================================

def processar_texto(texto):

    # ---------------------------------
    # 0. proteção contra NaN / None
    # ---------------------------------
    if not isinstance(texto, str):
        return ""

    texto = reconstruir(texto)
    texto = limpar_formatacao(texto)

    return texto


# =====================================================
# 4. APLICAR EM DATAFRAME
# =====================================================

def aplicar_pipeline(df):

    df["Introduction_clean"] = df["Introduction"].apply(processar_texto)
    df["Conclusion_clean"] = df["Conclusion"].apply(processar_texto)

    return df

In [ ]:
df = aplicar_pipeline(df)

In [ ]:
df.info()

In [ ]:
df.to_csv('dataset_de_amostras_final.csv', index=False)